# FinantalLM — Chat UI (Gradio) for Google Colab

A ChatGPT-style interface to test your model **while it trains**, loading checkpoints
straight from Google Drive. Run the cells top to bottom.

- Cell 4 **inspects** your `.pt` file and prints exactly what it contains (no assumptions).
- The model architecture is rebuilt automatically from the `model_config` stored inside the checkpoint.
- Switch between `latest.pt` / `step_3700.pt` live from the UI.


In [ ]:
# 1) Dependencies (torch is already on Colab)
!pip -q install gradio sentencepiece
import torch, gradio as gr
print('torch', torch.__version__, '| gradio', gr.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Paths — EDIT these if your layout differs
import os, glob

CKPT_DIR = '/content/drive/MyDrive/pretrain/checkpoints'   # folder with latest.pt / step_*.pt

# Tokenizer (SentencePiece .model) is NOT inside the checkpoint — locate it on Drive.
TOKENIZER_PATH = ''  # leave '' to auto-search, or set the full path explicitly
TOKENIZER_CANDIDATES = [
    '/content/drive/MyDrive/finantal_data/tokenizer/finantal_tokenizer.model',
    '/content/drive/MyDrive/pretrain/tokenizer/finantal_tokenizer.model',
    '/content/drive/MyDrive/tokenizer/finantal_tokenizer.model',
    '/content/drive/MyDrive/pretrain/finantal_tokenizer.model',
]

# --- checkpoints found ---
ckpt_files = sorted(glob.glob(os.path.join(CKPT_DIR, '*.pt')))
assert ckpt_files, f'No .pt files found in {CKPT_DIR}. Fix CKPT_DIR.'
print('Checkpoints found:')
for p in ckpt_files: print('  -', os.path.basename(p), f'({os.path.getsize(p)/1e6:.1f} MB)')

# --- tokenizer auto-search ---
if not TOKENIZER_PATH:
    for c in TOKENIZER_CANDIDATES:
        if os.path.exists(c):
            TOKENIZER_PATH = c; break
if not TOKENIZER_PATH:
    hits = glob.glob('/content/drive/MyDrive/**/*.model', recursive=True)
    TOKENIZER_PATH = hits[0] if hits else ''
print('\nTokenizer:', TOKENIZER_PATH or 'NOT FOUND -- set TOKENIZER_PATH manually')
assert TOKENIZER_PATH and os.path.exists(TOKENIZER_PATH), 'Set TOKENIZER_PATH to your finantal_tokenizer.model'

In [ ]:
# 4) INSPECT the checkpoint — see exactly what is stored (no assumptions)
import torch
_sample = os.path.join(CKPT_DIR, 'latest.pt')
if not os.path.exists(_sample): _sample = ckpt_files[0]
print('Inspecting:', _sample, '\n')

_ck = torch.load(_sample, map_location='cpu', weights_only=False)
print('type:', type(_ck).__name__)
if isinstance(_ck, dict):
    for k, v in _ck.items():
        if isinstance(v, dict):
            n_t = sum(1 for x in v.values() if torch.is_tensor(x))
            print(f'  {k:14s}: dict ({len(v)} entries, {n_t} tensors)')
        elif torch.is_tensor(v):
            print(f'  {k:14s}: tensor {tuple(v.shape)}')
        else:
            print(f'  {k:14s}: {type(v).__name__} = {str(v)[:80]}')
    if isinstance(_ck.get('model_config'), dict):
        print('\nmodel_config:')
        for k, v in _ck['model_config'].items(): print(f'    {k} = {v}')
else:
    print('Checkpoint is a bare state_dict (no wrapper).')

In [ ]:
# 5) Model architecture (identical to the training code so weights load exactly)
"""
Finantal decoder-only language model — implemented from scratch in PyTorch.

Architecture: Llama-style causal transformer
  - Token embedding (optionally tied to the output projection)
  - Pre-norm RMSNorm
  - Rotary Position Embeddings (RoPE)
  - Grouped-Query Attention (GQA) using F.scaled_dot_product_attention
  - SwiGLU feed-forward
  - Optional gradient checkpointing (essential for fitting larger models on a T4)

No HuggingFace `transformers` model classes, no pretrained weights. Everything
here is plain torch.nn so the training loop has full control.
"""

from __future__ import annotations

import json
import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    hidden_size: int = 1024
    intermediate_size: int = 2816
    num_hidden_layers: int = 24
    num_attention_heads: int = 16
    num_key_value_heads: int = 8
    head_dim: Optional[int] = None
    max_position_embeddings: int = 1024
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-5
    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    initializer_range: float = 0.02
    tie_word_embeddings: bool = True
    use_gradient_checkpointing: bool = True
    pad_token_id: int = 0
    unk_token_id: int = 1
    bos_token_id: int = 2
    eos_token_id: int = 3

    def __post_init__(self):
        if self.head_dim is None:
            assert self.hidden_size % self.num_attention_heads == 0, \
                "hidden_size must be divisible by num_attention_heads"
            self.head_dim = self.hidden_size // self.num_attention_heads
        assert self.num_attention_heads % self.num_key_value_heads == 0, \
            "num_attention_heads must be divisible by num_key_value_heads"

    @classmethod
    def from_json(cls, path: str) -> "ModelConfig":
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        # keep only fields the dataclass knows about (config JSON has extra metadata keys)
        fields = {f for f in cls.__dataclass_fields__}  # type: ignore[attr-defined]
        clean = {k: v for k, v in raw.items() if k in fields}
        return cls(**clean)

    def to_dict(self) -> dict:
        return {k: getattr(self, k) for k in self.__dataclass_fields__}  # type: ignore[attr-defined]


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # compute the norm in fp32 for stability, then cast back
        dtype = x.dtype
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.to(dtype)) * self.weight


def build_rope_cache(seq_len: int, head_dim: int, theta: float, device, dtype):
    """Precompute cos/sin tables for rotary embeddings: shape [seq_len, head_dim]."""
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)               # [seq_len, head_dim/2]
    emb = torch.cat((freqs, freqs), dim=-1)        # [seq_len, head_dim]
    return emb.cos().to(dtype), emb.sin().to(dtype)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q,k: [B, n_heads, T, head_dim]; cos/sin: [T, head_dim]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """[B, n_kv, T, hd] -> [B, n_kv*n_rep, T, hd] for grouped-query attention."""
    if n_rep == 1:
        return x
    b, n_kv, t, hd = x.shape
    return (
        x[:, :, None, :, :]
        .expand(b, n_kv, n_rep, t, hd)
        .reshape(b, n_kv * n_rep, t, hd)
    )


class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_heads = cfg.num_attention_heads
        self.n_kv_heads = cfg.num_key_value_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.head_dim = cfg.head_dim
        self.dropout = cfg.attention_dropout

        self.q_proj = nn.Linear(cfg.hidden_size, self.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.hidden_size, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.hidden_size, self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_heads * self.head_dim, cfg.hidden_size, bias=False)

    def forward(self, x, cos, sin):
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # Flash / memory-efficient SDPA with a causal mask — no [T,T] mask materialized.
        attn = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        attn = attn.transpose(1, 2).contiguous().view(b, t, -1)
        return self.o_proj(attn)


class SwiGLU(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.up_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.down_proj = nn.Linear(cfg.intermediate_size, cfg.hidden_size, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class DecoderLayer(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.input_norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.attn = Attention(cfg)
        self.post_norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.mlp = SwiGLU(cfg)
        self.res_dropout = nn.Dropout(cfg.residual_dropout)

    def forward(self, x, cos, sin):
        x = x + self.res_dropout(self.attn(self.input_norm(x), cos, sin))
        x = x + self.res_dropout(self.mlp(self.post_norm(x)))
        return x


class FinantalForCausalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden_size, padding_idx=cfg.pad_token_id)
        self.layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.num_hidden_layers)])
        self.norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

        if cfg.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

        self._rope_cache = {}  # (seq_len, device, dtype) -> (cos, sin)
        self.apply(self._init_weights)

        # scaled init for residual projections (GPT-2 style) — stabilizes deep nets
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=cfg.initializer_range / math.sqrt(2 * cfg.num_hidden_layers))

    def _init_weights(self, module):
        std = self.cfg.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].zero_()

    def _get_rope(self, seq_len, device, dtype):
        key = (seq_len, device, dtype)
        if key not in self._rope_cache:
            self._rope_cache[key] = build_rope_cache(
                seq_len, self.cfg.head_dim, self.cfg.rope_theta, device, dtype
            )
        return self._rope_cache[key]

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        b, t = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self._get_rope(t, x.device, x.dtype)

        use_ckpt = self.cfg.use_gradient_checkpointing and self.training
        for layer in self.layers:
            if use_ckpt:
                x = checkpoint(layer, x, cos, sin, use_reentrant=False)
            else:
                x = layer(x, cos, sin)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # shift so position i predicts token i+1
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)).float(),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=64, temperature=1.0, top_k=None, eos_token_id=None):
        """Lightweight sampler for sanity-checking a checkpoint (no KV cache — keep it short)."""
        self.eval()
        eos_token_id = self.cfg.eos_token_id if eos_token_id is None else eos_token_id
        max_ctx = self.cfg.max_position_embeddings
        for _ in range(max_new_tokens):
            ctx = input_ids[:, -max_ctx:]
            logits, _ = self.forward(ctx)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, nxt], dim=1)
            if eos_token_id is not None and (nxt == eos_token_id).all():
                break
        return input_ids

    def num_parameters(self, trainable_only: bool = True) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad or not trainable_only)


In [ ]:
# 6) Build model from a checkpoint + load tokenizer (robust key detection)
import sentencepiece as spm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def _find_state_dict(ck):
    if isinstance(ck, dict):
        for k in ('model', 'model_state_dict', 'state_dict'):
            if isinstance(ck.get(k), dict):
                return ck[k]
        if all(torch.is_tensor(v) for v in ck.values()):
            return ck
    raise ValueError('Could not find a model state_dict in this checkpoint.')

def _find_config(ck, state):
    if isinstance(ck, dict):
        for k in ('model_config', 'config'):
            if isinstance(ck.get(k), dict):
                return ck[k]
    # fallback: infer the essentials from tensor shapes
    emb = state.get('embed_tokens.weight')
    n_layers = 1 + max(int(k.split('.')[1]) for k in state if k.startswith('layers.'))
    print('[warn] no model_config in checkpoint -> inferring vocab/hidden/layers from weights')
    return {'vocab_size': emb.shape[0], 'hidden_size': emb.shape[1], 'num_hidden_layers': n_layers}

_MODEL_CACHE = {}

def load_model(ckpt_path):
    if ckpt_path in _MODEL_CACHE:
        return _MODEL_CACHE[ckpt_path]
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state = _find_state_dict(ck)
    cfg_d = _find_config(ck, state)
    cfg = ModelConfig(**{k: v for k, v in cfg_d.items() if k in ModelConfig.__dataclass_fields__})
    cfg.use_gradient_checkpointing = False           # inference only
    model = FinantalForCausalLM(cfg)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:    print('[load] missing keys:', missing[:6], '...' if len(missing) > 6 else '')
    if unexpected: print('[load] unexpected keys:', unexpected[:6], '...' if len(unexpected) > 6 else '')
    model.to(DEVICE).eval()
    step = ck.get('step') if isinstance(ck, dict) else None
    print(f'[load] {os.path.basename(ckpt_path)} | step={step} | params={model.num_parameters():,} | {DEVICE}')
    _MODEL_CACHE[ckpt_path] = (model, cfg)
    return model, cfg

# tokenizer
SP = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)
print('tokenizer vocab:', SP.get_piece_size(), '| bos', SP.bos_id(), '| eos', SP.eos_id())

# warm-load the default checkpoint
_default = os.path.join(CKPT_DIR, 'latest.pt')
if not os.path.exists(_default): _default = ckpt_files[0]
load_model(_default)

In [ ]:
# 7) Sampling generation (top_p / top_k / repetition_penalty / do_sample),
#    returns ONLY the newly generated text.
import torch.nn.functional as F

@torch.no_grad()
def generate(model, cfg, prompt, max_new_tokens=160, temperature=0.2, top_p=0.9,
             top_k=50, repetition_penalty=1.3, do_sample=True, stop_str='User:'):
    # do NOT prepend BOS: SFT data starts with '▁User'(26054), never BOS -> match training
    ids = SP.encode(prompt, out_type=int)
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    max_ctx = cfg.max_position_embeddings
    eos_id = cfg.eos_token_id
    new_tokens = []
    for _ in range(int(max_new_tokens)):
        logits, _ = model(x[:, -max_ctx:])
        logits = logits[:, -1, :].float()
        # repetition penalty over tokens already in the sequence
        if repetition_penalty and repetition_penalty != 1.0:
            for t in set(x[0].tolist()):
                logits[0, t] = logits[0, t] * repetition_penalty if logits[0, t] < 0 else logits[0, t] / repetition_penalty
        if do_sample:
            logits = logits / max(float(temperature), 1e-5)
            if top_k and top_k > 0:
                v, _ = torch.topk(logits, min(int(top_k), logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            if top_p and top_p < 1.0:
                s_logits, s_idx = torch.sort(logits, descending=True)
                cum = torch.cumsum(F.softmax(s_logits, dim=-1), dim=-1)
                remove = cum > top_p
                remove[..., 1:] = remove[..., :-1].clone(); remove[..., 0] = False
                logits = logits.masked_fill(remove.scatter(1, s_idx, remove), float('-inf'))
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        else:
            nxt = logits.argmax(dim=-1, keepdim=True)
        tok = int(nxt.item())
        x = torch.cat([x, nxt], dim=1)
        if eos_id is not None and tok == eos_id:
            break
        new_tokens.append(tok)
        if stop_str and stop_str in SP.decode(new_tokens):
            return SP.decode(new_tokens).split(stop_str)[0].strip()
    return SP.decode(new_tokens).strip()

In [ ]:
# 8) Chat plumbing + Gradio UI
# STATELESS: history is intentionally ignored. The model is single-turn trained
# ('User: <q> Assistant: <a>'); feeding accumulated history causes context bleed
# and off-topic answers. Each message is wrapped in the exact SFT template.
def respond(message, history, checkpoint, max_new_tokens, temperature, top_p,
            top_k, repetition_penalty, do_sample):
    model, cfg = load_model(os.path.join(CKPT_DIR, checkpoint))
    prompt = f'User: {message.strip()} Assistant:'
    return generate(model, cfg, prompt, max_new_tokens=max_new_tokens, temperature=temperature,
                    top_p=top_p, top_k=top_k, repetition_penalty=repetition_penalty, do_sample=do_sample)

_choices = [os.path.basename(p) for p in ckpt_files]
_default_choice = 'latest.pt' if 'latest.pt' in _choices else _choices[0]

demo = gr.ChatInterface(
    fn=respond,
    type='messages',
    title='FinantalLM',
    description='Chat with your financial LM checkpoints (loaded live from Google Drive).',
    additional_inputs=[
        gr.Dropdown(choices=_choices, value=_default_choice, label='Checkpoint'),
        gr.Slider(16, 512, value=160, step=8, label='max_new_tokens'),
        gr.Slider(0.1, 1.5, value=0.2, step=0.05, label='temperature (low=focused)'),
        gr.Slider(0.1, 1.0, value=0.9, step=0.05, label='top_p'),
        gr.Slider(0, 200, value=50, step=1, label='top_k (0 = off)'),
        gr.Slider(1.0, 2.0, value=1.3, step=0.05, label='repetition_penalty'),
        gr.Checkbox(value=True, label='do_sample'),
    ],
    additional_inputs_accordion=gr.Accordion('Generation settings', open=True),
)

demo.launch(share=True, debug=False)